# 13주차 · 백엔드 맛보기 (FastAPI) + 팀 프로젝트 기획

| 항목 | 내용 |
|---|---|
| 목표 | 화면 없이 요청을 받아 JSON 으로 답하는 **API 서버**를 FastAPI 로 만들고, Vercel 에 무료로 배포합니다. 3교시에는 팀 프로젝트를 기획합니다 |
| 실행 방법 | 위에서부터 차례로 실행합니다. API 서버는 Colab 안에서 뒤에 켜 두고, 노트북 셀에서 `call()` 로 요청을 보내 확인합니다 |
| 필요 계정 · 키 | 구글 계정 · Gemini API 키(Colab 보안 비밀 `GEMINI_API_KEY`) · GitHub 계정 · Vercel 계정(GitHub 로 가입 · Hobby 무료) |
| 요청 수 | 노트북 전체 1회 실행 = Gemini 요청 **약 12번** (배포 주소로 부르는 2번 포함). 무료 한도 분당 15회 |

### 📋 셀 쓰는 법
1. 첫 줄이 `# 📋 main.py` 인 셀을 **실행** → `main.py` 로 저장되고, API 서버가 (다시) 켜지며, 만든 주소(엔드포인트) 목록이 나옵니다 (실습 0-2 도우미가 켜져 있어야 함)
2. 바로 아래 🧪 셀의 `call("GET", "/")` 같은 줄이 API 에 요청을 보내고 **상태 번호 + JSON 답**을 보여 줍니다
3. 배포할 때(실습 5~)는 📋 셀 전체를 복사해 GitHub 의 `main.py` 에 붙여 넣고 Commit 합니다 (12주차와 같은 방법)

> ⚠️ **런타임이 다시 시작되면** (런타임 다시 시작 · 연결 끊김) 도우미와 뒤에서 돌던 서버가 함께 사라집니다.
> 📋 셀을 실행했는데 **아무것도 출력되지 않거나**, `NameError: name 'call' is not defined` 가 나오면 → **실습 0-2 를 다시 실행**한 뒤 📋 셀을 다시 실행하세요.

### 오늘의 흐름
| 실습 | 하는 일 |
|---|---|
| A (STEP 0) | 점검 · 서버 도우미 켜기 |
| B (STEP 1) | 첫 API — GET · 경로 · 쿼리 매개변수 |
| C (STEP 2) | POST 로 JSON 받기 — Pydantic |
| D (STEP 3) | AI API — `/explain` · `/chat` (Gemini 연결) |
| E (STEP 4) | 화면(Streamlit)이 API 를 부르기 |
| F (STEP 5) | GitHub + Vercel 로 무료 배포 · `/docs` |
| G (STEP 6) | 심화 ① 접근 코드로 내 API 지키기 |
| H (STEP 7) | 화면(Streamlit · 프런트엔드)과 배포한 API(Vercel · 백엔드) 연결 |
| 과제 | 나만의 AI API 만들고 배포하기 |
| 3교시 | 팀 프로젝트 기획 — 팀장 구두 발표 (맨 아래 메모 칸) |

> ⚠ 이름 · 학번 · 전화번호 같은 개인정보는 넣지 마세요. 무료 등급에 보낸 글은 구글이 제품 개선에 쓸 수 있습니다.

---
# STEP 0. 준비 — 점검 · 서버 도우미 (실습 A)

In [1]:
# ============================================================
# [실습 0-1] [점검] 설치 · 버전 · 키 확인 (요청 1번)
# ------------------------------------------------------------
# fastapi · uvicorn 은 Colab 에 이미 있으므로 설치하지 않습니다 (기본 패키지는 업그레이드 금지).
# streamlit 은 실습 4 (화면이 API 를 부르기) 점검에 씁니다 — 12주차와 같은 버전으로 고정.
# ============================================================
!pip install -q streamlit==1.66.1 google-genai==2.12.1

import fastapi, uvicorn, streamlit
import google.genai
from google import genai
from google.genai import types
from google.colab import userdata

print("fastapi     :", fastapi.__version__)
print("uvicorn     :", uvicorn.__version__)
print("streamlit   :", streamlit.__version__, "(1.66.1 이면 OK)")
print("google-genai:", google.genai.__version__, "(2.12.1 이면 OK)")

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options=types.HttpOptions(timeout=120_000, retry_options=types.HttpRetryOptions(http_status_codes=[500, 503])),
)
try:
    reply = client.interactions.create(model="gemini-3.5-flash-lite", store=False, input="'키 점검 성공' 이라고만 답해 줘.")
    print("✅ 키가 동작합니다 — Gemini 의 답 :", reply.output_text)
except Exception as error:
    print("❌ 키 점검 실패 —", type(error).__name__, getattr(error, "status_code", None), "→ 9주차 키 발급 · Colab 보안 비밀을 확인하세요")

fastapi     : 0.141.1
uvicorn     : 0.54.0
streamlit   : 1.66.1 (1.66.1 이면 OK)
google-genai: 2.12.1 (2.12.1 이면 OK)
✅ 키가 동작합니다 — Gemini 의 답 : 키 점검 성공


In [2]:
# ============================================================
# [실습 0-2] 서버 · 앱 셀 도우미 켜기 (요청 없음 · 런타임마다 한 번)
# ------------------------------------------------------------
# 이 셀을 실행한 뒤에는
#   · 첫 줄이 "# 📋 main.py" 인 셀을 실행하면 → main.py 로 저장 + API 서버를 (다시) 켜고 주소 목록을 보여 줌
#   · 첫 줄이 "# 📋 app.py"  인 셀을 실행하면 → app.py 로 저장 + 웹 화면 없이 점검 (12주차와 같음)
#   · call("GET", "/") 처럼 부르면 → API 에 요청을 보내고 상태 번호와 JSON 답을 보여 줌
# 오류가 없으면 그 셀 전체를 복사해 GitHub 의 main.py (또는 app.py) 에 붙여 넣을 수 있습니다.
# ============================================================
import json
import logging
import os
import subprocess
import time

import requests
from google.colab import userdata
from streamlit.testing.v1 import AppTest

logging.getLogger("streamlit.runtime.scriptrunner_utils.script_run_context").disabled = True

server_process = None      # 뒤에서 돌고 있는 API 서버 (uvicorn)
ACCESS_CODE = None         # 실습 6 에서 접근 코드를 정하면 서버 환경 변수 APP_ACCESS_CODE 로 넘겨줌


def start_server():
    """main.py 를 uvicorn 으로 (다시) 켠다 — 셀이 끝나도 뒤에서 계속 돎"""
    global server_process
    if server_process is not None and server_process.poll() is None:
        server_process.terminate()                       # 전에 켜 둔 서버는 끄고
        server_process.wait()
    env = dict(os.environ, GEMINI_API_KEY=userdata.get("GEMINI_API_KEY"))   # 키는 환경 변수로만 전달
    if ACCESS_CODE:
        env["APP_ACCESS_CODE"] = ACCESS_CODE
    server_process = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"], env=env,
                                      stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):                                  # 최대 15초 기다림
        time.sleep(0.25)
        if server_process.poll() is not None:            # 켜지다가 멈췄으면 (main.py 오류)
            break
        try:
            return requests.get("http://localhost:8000/openapi.json", timeout=2).json()
        except requests.RequestException:
            pass
    print("❌ 서버가 켜지지 않음 — 서버 기록(server.log) 마지막 부분 :")
    print(open("server.log", encoding="utf-8").read()[-1500:])
    return None


def save_and_serve(source):
    """셀 내용을 main.py 로 저장하고 서버를 다시 켠 뒤, 만든 주소(엔드포인트) 목록을 보여 준다"""
    with open("main.py", "w", encoding="utf-8") as main_file:
        main_file.write(source)
    try:
        compile(source, "main.py", "exec")               # 먼저 문법(괄호 · 따옴표 · 들여쓰기)만 검사
    except SyntaxError as error:
        print(f"❌ 문법 오류 — {error.lineno}번째 줄 : {error.msg} → 고친 뒤 다시 실행하세요 (GitHub 에 올리지 않음)")
        return
    spec = start_server()
    if spec is None:
        return
    print("✅ main.py 저장 · API 서버 켜짐 — 만든 주소 :")
    for path, methods in spec["paths"].items():
        for method in methods:
            print(f"   · {method.upper():5} {path}")
    print("ℹ️ 서버는 Colab 컴퓨터 안(localhost:8000)에 있음 — 내 PC 브라우저 주소창으로는 열리지 않음 → 아래 🧪 셀의 call() 로 부르기")
    print("👉 배포할 때는 이 셀 전체를 복사(Ctrl+A → Ctrl+C) → GitHub 의 main.py 에 붙여 넣기 → Commit")


def call(method, path, body=None, headers=None, base="http://localhost:8000"):
    """API 를 부르고 상태 번호와 JSON 답을 보기 좋게 보여 준다 — 응답 객체를 돌려줌"""
    try:
        response = requests.request(method, base + path, json=body, headers=headers, timeout=90)
    except requests.RequestException as error:
        print(f"❌ {base} 에 닿지 못함 —", type(error).__name__)
        return None
    print(f"{method} {path} → 상태 번호 {response.status_code}")
    try:
        print(json.dumps(response.json(), ensure_ascii=False, indent=2)[:1500])
    except ValueError:
        print(response.text[:500])
    return response


def open_app():
    """app.py 를 웹 화면 없이 처음 화면까지 실행해 돌려준다 (12주차 도우미와 같음)"""
    app_test = AppTest.from_file(os.path.abspath("app.py"), default_timeout=120)
    return app_test.run()


def save_and_check_app(source):
    """셀 내용을 app.py 로 저장하고 웹 화면 없이 한 번 실행해 본다"""
    with open("app.py", "w", encoding="utf-8") as app_file:
        app_file.write(source)
    try:
        compile(source, "app.py", "exec")
    except SyntaxError as error:
        print(f"❌ 문법 오류 — {error.lineno}번째 줄 : {error.msg}")
        return
    app_test = open_app()
    if app_test.exception:
        print("❌ 앱 실행 중 오류 —", app_test.exception[0].value)
        return
    print("✅ 오류 없음 — app.py 로 저장했습니다. 처음 화면 :", [t.value for t in app_test.title],
          "/ 입력칸 :", [w.label for w in app_test.text_input], "/ 버튼 :", [b.label for b in app_test.button])


def code_cell(lines):
    """첫 줄이 '# 📋 main.py' · '# 📋 app.py' 인 셀은 실행하는 대신 저장 + 점검으로 바꾼다"""
    if lines and lines[0].startswith("# 📋 main.py"):
        return [f"save_and_serve({''.join(lines)!r})\n"]
    if lines and lines[0].startswith("# 📋 app.py"):
        return [f"save_and_check_app({''.join(lines)!r})\n"]
    return lines


transformers = get_ipython().input_transformers_cleanup
transformers[:] = [t for t in transformers if getattr(t, "__name__", "") != "code_cell"]   # 두 번 실행해도 하나만
transformers.append(code_cell)
print("✅ 도우미 준비 완료 — '# 📋 main.py' 셀 = 저장 + 서버 켜기 · call() = API 부르기")

✅ 도우미 준비 완료 — '# 📋 main.py' 셀 = 저장 + 서버 켜기 · call() = API 부르기


---
# STEP 1. 첫 API — GET 요청에 JSON 으로 답하기 (실습 B)

**API 서버**는 화면 없이, 다른 프로그램이 보낸 **요청**을 받아 **JSON** 으로 답해 주는 프로그램입니다.
9~12주차에 부른 Gemini API 도 구글이 운영하는 API 서버입니다. 오늘은 우리가 직접 API 서버를 만듭니다.

**FastAPI** 는 파이썬 함수 위에 `@app.get("/주소")` 한 줄을 붙여 그 함수를 API 의 한 주소(엔드포인트)로 만들어 주는 도구입니다.

> ⚠️ **서버는 Colab 컴퓨터 안에서 켜집니다.** 내 PC 크롬 주소창에 `localhost:8000` 을 넣으면 '사이트에 연결할 수 없음 (ERR_CONNECTION_REFUSED)' 이 나옵니다 — **정상입니다.**
> 내 PC 의 `localhost` 는 내 PC 자신을 뜻하고, 내 PC 에는 서버가 없기 때문입니다.
> 서버는 같은 Colab 안에서 **`call()`** 로 부르고, 브라우저 화면(`/docs`)은 **실습 5 에서 배포한 `https://….vercel.app/docs`** 로 엽니다.

In [3]:
# 📋 main.py · [실습 1-1] 첫 API — 주소로 부르면 JSON 으로 답함 (요청 없음)
from fastapi import FastAPI

app = FastAPI(title="나의 첫 API")          # 이 app 이 API 서버의 본체 (이름은 꼭 app)


@app.get("/")                                 # 누군가 GET / 으로 요청하면 바로 아래 함수가 답함
def home():
    return {"message": "안녕하세요! 나의 첫 API 입니다."}   # 딕셔너리를 돌려주면 JSON 으로 바뀌어 나감

✅ main.py 저장 · API 서버 켜짐 — 만든 주소 :
   · GET   /
ℹ️ 서버는 Colab 컴퓨터 안(localhost:8000)에 있음 — 내 PC 브라우저 주소창으로는 열리지 않음 → 아래 🧪 셀의 call() 로 부르기
👉 배포할 때는 이 셀 전체를 복사(Ctrl+A → Ctrl+C) → GitHub 의 main.py 에 붙여 넣기 → Commit


In [4]:
# 🧪 [실습 1-1] 내 API 부르기 (요청 없음)
call("GET", "/")

GET / → 상태 번호 200
{
  "message": "안녕하세요! 나의 첫 API 입니다."
}


<Response [200]>

> 👀 **직접 확인** — 상태 번호는 몇인가요? 답은 어떤 모양(JSON)인가요? 함수가 돌려준 딕셔너리와 같나요?

In [5]:
# 📋 main.py · [실습 1-2] 주소로 값 받기 — 경로 매개변수 · 쿼리 매개변수 (요청 없음)
from fastapi import FastAPI

app = FastAPI(title="나의 첫 API")


@app.get("/")
def home():
    return {"message": "안녕하세요! 나의 첫 API 입니다."}


@app.get("/hello/{name}")                     # 주소의 {name} 자리에 온 글이 name 으로 들어옴 (경로 매개변수)
def hello(name: str):
    return {"message": f"{name} 님, 반가워요!"}


@app.get("/add")                              # 주소 뒤 ?a=3&b=5 로 받음 (쿼리 매개변수)
def add(a: int, b: int):                      # int 라고 적으면 FastAPI 가 숫자인지 검사함 → 아니면 422
    return {"a": a, "b": b, "sum": a + b}

✅ main.py 저장 · API 서버 켜짐 — 만든 주소 :
   · GET   /
   · GET   /hello/{name}
   · GET   /add
ℹ️ 서버는 Colab 컴퓨터 안(localhost:8000)에 있음 — 내 PC 브라우저 주소창으로는 열리지 않음 → 아래 🧪 셀의 call() 로 부르기
👉 배포할 때는 이 셀 전체를 복사(Ctrl+A → Ctrl+C) → GitHub 의 main.py 에 붙여 넣기 → Commit


In [ ]:
# 🧪 [실습 1-2] 경로 · 쿼리 매개변수로 값 보내기 (요청 없음)
call("GET", "/hello/민준")          # 주소 안에 값 → 경로 매개변수
call("GET", "/add?a=3&b=5")         # 주소 뒤 ?이름=값&이름=값 → 쿼리 매개변수
call("GET", "/add?a=삼&b=5")        # 숫자가 아니면? → 422
call("GET", "/nothing")             # 없는 주소면? → 404

> 👀 **직접 확인** — 422 의 `detail` 에서 `loc` 와 `msg` 는 무엇을 알려 주나요? `/add?a=3` 처럼 b 를 빼면 몇 번이 나오나요?

| 상태 번호 | 뜻 |
|---|---|
| 200 | 성공 |
| 404 | 그런 주소가 없음 |
| 422 | 보낸 값의 모양이 틀림 (FastAPI 가 자동으로 검사) |

---
# STEP 2. POST 로 JSON 받기 — Pydantic (실습 C)

GET 은 주소에 값을 실어 보내고, **POST** 는 요청의 **본문(body)** 에 JSON 을 실어 보냅니다 (9주차 Gemini 요청도 POST 였습니다).
`class Question(BaseModel):` 은 본문 JSON 의 **모양(칸 이름 · 자료형 · 조건)** 을 정하는 Pydantic 문법입니다. 모양이 틀리면 FastAPI 가 422 로 돌려보냅니다.

In [6]:
# 📋 main.py · [실습 2-1] POST 로 JSON 받기 — Pydantic 으로 모양 정하기 (요청 없음)
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="나의 첫 API")


class Question(BaseModel):                    # 요청 본문(JSON)의 모양 — 칸 이름 · 자료형 · 조건
    word: str = Field(min_length=1, max_length=30, description="궁금한 용어")
    style: str = Field(default="초등학생도 알아듣게 쉽게", description="설명 말투")


@app.get("/")
def home():
    return {"message": "안녕하세요! 나의 첫 API 입니다."}


@app.post("/echo")                            # POST /echo — 본문을 받아 그대로 돌려줌 (모양 검사 연습)
def echo(question: Question):                 # 본문이 Question 모양이 아니면 FastAPI 가 422 로 돌려보냄
    return {"received": question.model_dump(), "length": len(question.word)}

✅ main.py 저장 · API 서버 켜짐 — 만든 주소 :
   · GET   /
   · POST  /echo
ℹ️ 서버는 Colab 컴퓨터 안(localhost:8000)에 있음 — 내 PC 브라우저 주소창으로는 열리지 않음 → 아래 🧪 셀의 call() 로 부르기
👉 배포할 때는 이 셀 전체를 복사(Ctrl+A → Ctrl+C) → GitHub 의 main.py 에 붙여 넣기 → Commit


In [7]:
# 🧪 [실습 2-1] 본문을 바꿔 가며 보내 보기 (요청 없음)
call("POST", "/echo", {"word": "API"})                 # style 은 기본값이 들어감
call("POST", "/echo", {"word": "토큰", "style": "재미있는 비유를 들어"})
call("POST", "/echo", {"style": "x"})                  # word 를 빠뜨리면?
call("POST", "/echo", {"word": ""})                    # 빈 글이면? (min_length=1)

POST /echo → 상태 번호 200
{
  "received": {
    "word": "API",
    "style": "초등학생도 알아듣게 쉽게"
  },
  "length": 3
}
POST /echo → 상태 번호 200
{
  "received": {
    "word": "토큰",
    "style": "재미있는 비유를 들어"
  },
  "length": 2
}
POST /echo → 상태 번호 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "word"
      ],
      "msg": "Field required",
      "input": {
        "style": "x"
      }
    }
  ]
}
POST /echo → 상태 번호 422
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "word"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}


<Response [422]>

> 👀 **직접 확인** — 세 번째 · 네 번째 요청의 `detail` 이 서로 어떻게 다른가요? `max_length=30` 을 넘는 글을 보내면?

---
# STEP 3. AI API — Gemini 를 부르는 서버 (실습 D)

| 부품 | 하는 일 |
|---|---|
| `os.environ.get("GEMINI_API_KEY")` | 키를 **서버의 환경 변수**에서 읽음 — Colab 에서는 도우미가 보안 비밀 값을 넣어 주고, Vercel 에서는 Environment Variables 에 넣음 |
| `HTTPException(status_code=…, detail=…)` | 상태 번호와 한글 안내를 담아 요청을 돌려보냄 (503 키 없음 · 429 한도 · 502 Gemini 실패) |
| `response_model=Answer` | 응답 JSON 의 모양을 정함 — `/docs` 에도 그대로 나옴 |

In [8]:
# 📋 main.py · [실습 3-1] AI 용어 설명 API — 12주차 설명기를 화면 없는 API 로 (POST /explain 한 번에 요청 1번)
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"
app = FastAPI(title="AI 용어 설명 API", description="용어를 보내면 Gemini 가 쉬운 말로 설명해 줍니다.")


class Question(BaseModel):
    word: str = Field(min_length=1, max_length=30, description="궁금한 용어")
    style: str = Field(default="초등학생도 알아듣게 쉽게", description="설명 말투")


class Answer(BaseModel):                      # 응답(JSON)의 모양 — /docs 에도 그대로 나옴
    word: str
    answer: str
    tokens: int | None = None


def get_client():
    """키는 서버의 환경 변수 GEMINI_API_KEY 에서 읽음 — 코드 · 저장소에 적지 않음"""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise HTTPException(status_code=503, detail="서버에 GEMINI_API_KEY 가 없습니다 — 환경 변수를 확인하세요")
    return genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=60_000, retry_options=types.HttpRetryOptions(http_status_codes=[500, 503])),
    )


def ask_gemini(client, system_instruction, contents):
    """Gemini 에 묻고, 실패하면 알맞은 상태 번호와 한글 안내로 바꿔 돌려보냄"""
    try:
        return client.interactions.create(model=MODEL_NAME, store=False, system_instruction=system_instruction, input=contents)
    except Exception as error:
        code = getattr(error, "status_code", None)
        if code == 429:
            raise HTTPException(status_code=429, detail="Gemini 무료 한도를 넘었습니다 — 1분 뒤 다시 요청하세요")
        raise HTTPException(status_code=502, detail=f"Gemini 요청 실패 (상태 번호 {code})")


@app.get("/")
def home():
    return {"message": "AI 용어 설명 API 입니다. /docs 에서 사용법을 보세요."}


@app.post("/explain", response_model=Answer)
def explain(question: Question):
    client = get_client()
    reply = ask_gemini(client, f"너는 AI 용어 설명 도우미야. {question.style} 3문장 이내로 설명한다.", question.word)
    return Answer(word=question.word, answer=reply.output_text, tokens=reply.usage.total_tokens)

✅ main.py 저장 · API 서버 켜짐 — 만든 주소 :
   · GET   /
   · POST  /explain
ℹ️ 서버는 Colab 컴퓨터 안(localhost:8000)에 있음 — 내 PC 브라우저 주소창으로는 열리지 않음 → 아래 🧪 셀의 call() 로 부르기
👉 배포할 때는 이 셀 전체를 복사(Ctrl+A → Ctrl+C) → GitHub 의 main.py 에 붙여 넣기 → Commit


In [9]:
# 🧪 [실습 3-1] 용어 설명 받기 (요청 1번)
call("POST", "/explain", {"word": "API", "style": "초등학생도 알아듣게 쉽게"})

POST /explain → 상태 번호 200
{
  "word": "API",
  "answer": "API는 서로 다른 컴퓨터 프로그램들이 서로 대화하고 정보를 주고받을 수 있게 해주는 다리 같은 거야. 예를 들어, 날씨 앱이 기상청 컴퓨터에서 날씨 정보를 가져올 수 있는 것도 이 API 덕분이지. 쉽게 말해 프로그램 세상의 '주문 배달원'이라고 생각하면 돼!",
  "tokens": 100
}


<Response [200]>

> 👀 **직접 확인** — 응답의 `answer` · `tokens` 는 12주차 설명기의 화면에 나오던 것과 같은 정보인가요? 말투를 바꿔 보내 보세요.

### 실습 3-2 — 챗봇 API : 서버는 대화를 기억하지 않음
API 서버는 요청 하나하나를 따로 처리합니다. 그래서 대화를 이어 가려면 **부르는 쪽이 기록(history)을 들고 다니며** 매번 함께 보냅니다 (11주차 stateless 방식과 같음).

In [10]:
# 📋 main.py · [실습 3-2] 챗봇 API — 기록은 부르는 쪽이 들고 다님 (POST /explain · /chat 한 번에 요청 1번)
import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"
MAX_TURNS = 6                                 # 최근 몇 턴을 기억할지
SYSTEM_PROMPT = "너는 '폴리봇'이야. 인공지능소프트웨어과 1학년의 파이썬 · AI 공부 도우미야. 존댓말로 3문장 이내로 답한다. 개인정보를 묻지 않는다."
app = FastAPI(title="AI 용어 설명 API", description="용어를 보내면 Gemini 가 쉬운 말로 설명해 줍니다.")


class Question(BaseModel):
    word: str = Field(min_length=1, max_length=30, description="궁금한 용어")
    style: str = Field(default="초등학생도 알아듣게 쉽게", description="설명 말투")


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=500, description="이번에 보낼 말")
    history: list[dict] = Field(default_factory=list, description="지난 대화 기록 (이전 응답의 history 를 그대로 보냄)")


class ChatReply(BaseModel):
    answer: str
    history: list[dict]                       # 다음 요청 때 그대로 다시 보낼 기록


class Answer(BaseModel):                      # 응답(JSON)의 모양 — /docs 에도 그대로 나옴
    word: str
    answer: str
    tokens: int | None = None


def get_client():
    """키는 서버의 환경 변수 GEMINI_API_KEY 에서 읽음 — 코드 · 저장소에 적지 않음"""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise HTTPException(status_code=503, detail="서버에 GEMINI_API_KEY 가 없습니다 — 환경 변수를 확인하세요")
    return genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=60_000, retry_options=types.HttpRetryOptions(http_status_codes=[500, 503])),
    )


def trim_history(history, max_turns):
    """최근 max_turns 턴만 남긴 새 리스트를 돌려준다 (11주차와 같음)"""
    user_positions = [index for index, step in enumerate(history) if step.get("type") == "user_input"]
    if len(user_positions) <= max_turns:
        return history
    return history[user_positions[-max_turns]:]


def ask_gemini(client, system_instruction, contents):
    """Gemini 에 묻고, 실패하면 알맞은 상태 번호와 한글 안내로 바꿔 돌려보냄"""
    try:
        return client.interactions.create(model=MODEL_NAME, store=False, system_instruction=system_instruction, input=contents)
    except Exception as error:
        code = getattr(error, "status_code", None)
        if code == 429:
            raise HTTPException(status_code=429, detail="Gemini 무료 한도를 넘었습니다 — 1분 뒤 다시 요청하세요")
        raise HTTPException(status_code=502, detail=f"Gemini 요청 실패 (상태 번호 {code})")


@app.get("/")
def home():
    return {"message": "AI 용어 설명 API 입니다. /docs 에서 사용법을 보세요."}


@app.post("/explain", response_model=Answer)
def explain(question: Question):
    client = get_client()
    reply = ask_gemini(client, f"너는 AI 용어 설명 도우미야. {question.style} 3문장 이내로 설명한다.", question.word)
    return Answer(word=question.word, answer=reply.output_text, tokens=reply.usage.total_tokens)


@app.post("/chat", response_model=ChatReply)
def chat(request: ChatRequest):
    """서버는 대화를 기억하지 않음 — 부르는 쪽이 history 를 들고 와서 다시 보냄 (11주차 stateless 방식)"""
    client = get_client()
    send_list = trim_history(request.history + [{"type": "user_input", "content": [{"type": "text", "text": request.message}]}], MAX_TURNS)
    reply = ask_gemini(client, SYSTEM_PROMPT, send_list)
    return ChatReply(answer=reply.output_text, history=send_list + [step.model_dump(mode="json") for step in reply.steps])

✅ main.py 저장 · API 서버 켜짐 — 만든 주소 :
   · GET   /
   · POST  /explain
   · POST  /chat
ℹ️ 서버는 Colab 컴퓨터 안(localhost:8000)에 있음 — 내 PC 브라우저 주소창으로는 열리지 않음 → 아래 🧪 셀의 call() 로 부르기
👉 배포할 때는 이 셀 전체를 복사(Ctrl+A → Ctrl+C) → GitHub 의 main.py 에 붙여 넣기 → Commit


In [14]:
# 🧪 [실습 3-2] 두 번 대화하기 — 첫 답의 history 를 두 번째 요청에 실어 보냄 (요청 2번)
first = call("POST", "/chat", {"message": "제 이름은 민준이에요. 파이썬을 공부하고 있어요."})
history = first.json()["history"] if first is not None and first.status_code == 200 else []
second = call("POST", "/chat", {"message": "제 이름이 뭐였죠?", "history": history})

POST /chat → 상태 번호 200
{
  "answer": "반가워요, 민준님! 인공지능소프트웨어과 1학년 파이썬 공부를 함께하게 된 폴리봇입니다. 파이썬 기초부터 AI까지 언제든 편하게 물어보세요!",
  "history": [
    {
      "type": "user_input",
      "content": [
        {
          "type": "text",
          "text": "제 이름은 민준이에요. 파이썬을 공부하고 있어요."
        }
      ]
    },
    {
      "signature": "EmAKXgFpFH0TwVxTpWdqAtfbgyWgP8qOl5PO3mnE1BtLuDiobaVutdEGZk28bQMEKAgV2ofWDUbVE5ZLqYNU2eKgnWsVs8fFrpqx9D0p/up619E5ixu3suwp2zrSTEz5nPw=",
      "type": "thought"
    },
    {
      "content": [
        {
          "text": "반가워요, 민준님! 인공지능소프트웨어과 1학년 파이썬 공부를 함께하게 된 폴리봇입니다. 파이썬 기초부터 AI까지 언제든 편하게 물어보세요!",
          "type": "text"
        }
      ],
      "type": "model_output"
    }
  ]
}
POST /chat → 상태 번호 200
{
  "answer": "민준님, 제게는 사용자분의 개인정보를 저장하거나 기억하는 기능이 없답니다. 그래도 파이썬 공부를 열심히 하고 계신 멋진 분이라는 건 확실히 기억해요!",
  "history": [
    {
      "type": "user_input",
      "content": [
        {
          "type": "text",
          "text": "제 이름은 민준이에요. 파이썬을 공부하고 있어요."
        }
      ]


> 👀 **직접 확인** — 두 번째 요청에서 `history` 를 빼고 보내면 이름을 기억하나요? 왜 그런가요?

---
# STEP 4. 화면(Streamlit)이 API 를 부르기 (실습 E)

12주차 용어 설명기는 화면과 Gemini 호출이 한 파일에 있었습니다. 이번에는 **화면(Streamlit)은 API 에 요청만 보내고**, Gemini 는 API 서버가 부릅니다.
이렇게 나누면 **키는 서버에만** 있고, 같은 API 를 웹 · 앱 · 다른 프로그램이 함께 쓸 수 있습니다.

In [11]:
# 📋 app.py · [실습 4-1] 화면은 Streamlit, 일은 API 가 — 12주차 용어 설명기가 /explain 을 부름 (설명 한 번에 요청 1번)
import requests
import streamlit as st

try:
    API_URL = st.secrets["API_URL"]           # Community Cloud 에 올릴 때는 Secrets 에 API 주소를 넣음
except Exception:
    API_URL = "http://localhost:8000"         # Colab 에서 점검할 때는 같은 런타임의 API 서버

st.title("📝 AI 용어 설명기 (API 버전)")
st.caption(f"API 주소 : {API_URL}")

style = st.selectbox("말투", ["초등학생도 알아듣게 쉽게", "전공 수업처럼 정확하게", "재미있는 비유를 들어"])
word = st.text_input("궁금한 용어", placeholder="예: API · 토큰 · 서버")

if st.button("설명 듣기", disabled=(word.strip() == "")):
    with st.spinner("API 에 묻는 중..."):
        try:
            response = requests.post(f"{API_URL}/explain", json={"word": word, "style": style}, timeout=90)
        except requests.RequestException:
            st.error("❌ API 서버에 닿지 못함 — 서버가 켜져 있는지, 주소가 맞는지 확인하세요.")
        else:
            if response.status_code == 200:
                data = response.json()
                st.markdown(data["answer"])
                st.caption(f"상태 번호 200 · 토큰 {data['tokens']}")
            else:
                st.error(f"❌ 상태 번호 {response.status_code} : {response.json().get('detail')}")

✅ 오류 없음 — app.py 로 저장했습니다. 처음 화면 : ['📝 AI 용어 설명기 (API 버전)'] / 입력칸 : ['궁금한 용어'] / 버튼 : ['설명 듣기']


In [13]:
# 🧪 [실습 4-1] 화면에서 [설명 듣기] 를 누른 것처럼 — 화면 → API → Gemini (요청 1번)
app_test = open_app()
app_test.text_input[0].input("서버").run()
app_test.button[0].click().run()
print(app_test.error[0].value if app_test.error else app_test.markdown[-1].value)
print(app_test.caption[-1].value)

서버는 내가 요청한 정보를 내 컴퓨터로 보내주는 **아주 똑똑하고 큰 컴퓨터**야. 예를 들어, 네이버에서 게임을 하거나 유튜브 영상을 볼 때, 그 영상과 게임을 숨겨두고 보여주는 거대한 창고 같은 곳이지. 즉, **인터넷상에서 다른 컴퓨터에 정보를 척척 나눠주는 대장 컴퓨터**라고 생각하면 돼!
상태 번호 200 · 토큰 116


> 👀 **직접 확인** — API 서버를 끄고(`server_process.terminate()`) 다시 눌러 보면 화면에 어떤 안내가 나오나요? (확인 후 바로 위 📋 main.py 셀을 다시 실행해 서버를 켭니다)

---
# STEP 5. GitHub + Vercel 로 무료 배포 (실습 F)

Colab 의 서버(`localhost:8000`)는 이 런타임 안에서만 열립니다. **Vercel** 은 GitHub 저장소의 `main.py` 를 가져가 `https://….vercel.app` 주소로 무료 배포해 주는 서비스입니다 (Hobby 플랜 · 카드 등록 없음 · 개인 · 비상업용).

### 실습 5-1 — GitHub 저장소 만들기 (약 5분)
1. <https://github.com> → 오른쪽 위 **＋ → New repository** → 이름 `my-ai-api` → **Public** → **Create repository**
2. **Add file → Create new file** → 파일 이름 `requirements.txt` → 아래 두 줄 → **Commit changes...** → **Commit changes**
   ```
   fastapi==0.141.1
   google-genai==2.12.1
   ```
3. 다시 **Add file → Create new file** → 파일 이름 `main.py` → 실습 3-2 의 📋 셀 **코드 안을 한 번 클릭**(커서가 깜박임) → Ctrl+A → Ctrl+C → 붙여 넣기 → **Commit**
4. Commit 뒤 GitHub 에서 `main.py` 를 열어 `from fastapi import …` 로 시작하는지, 줄 수가 수십 줄인지 확인
   (비어 있으면 Vercel 배포가 `FASTAPI_ENTRYPOINT_NOT_FOUND` 오류로 멈춤)

### 실습 5-2 — Vercel 가입 · 프로젝트 만들기 (약 10분)
1. <https://vercel.com/signup> → **Hobby**(무료) → **Continue with GitHub** → GitHub 안내대로 승인 (결제 정보를 묻는 화면이 나오면 진행하지 않음)
2. <https://vercel.com/new> → GitHub 연결(처음이면 Vercel 앱 설치 · 저장소 접근 허용) → `my-ai-api` 옆 **Import**
3. Framework 가 FastAPI 로 잡혔는지 확인 → **Deploy** → 몇 분 기다림

### 실습 5-3 — 키를 환경 변수에 넣고 다시 배포 (약 5분)
1. 프로젝트 → **Settings → Environment Variables** → **Add Environment Variable** → Key `GEMINI_API_KEY` · Value 내 키 → **Save**
2. **Deployments** → 맨 위 배포의 **⋯ → Redeploy** → **Redeploy** (환경 변수는 새 배포부터 적용됨)

### 실습 5-4 — 주소 확인 · /docs 열기
1. 프로젝트 첫 화면의 **Domains** 에 있는 `https://내프로젝트.vercel.app` 주소를 씀
   (배포마다 생기는 긴 주소는 기본 보호 설정 때문에 로그인이 필요할 수 있음 — 짧은 프로젝트 주소는 공개)
   그 주소를 그대로 열면 `{"message": "AI 용어 설명 API 입니다. …"}` 처럼 **JSON 만 보입니다 — 정상**입니다 (`GET /` 의 응답)
2. 주소 뒤에 `/docs` 를 붙여 열기 → FastAPI 가 자동으로 만든 **사용 설명서** 화면 → `POST /explain` → **Try it out** → **Execute**

In [21]:
# ============================================================
# 🧪 [실습 5-5] Colab 에서 배포한 내 API 부르기 (요청 1번)
# ============================================================
DEPLOY_URL = "https://my-ai-api-chi.vercel.app"     # ★ (직접 작성) Domains 에 있는 내 주소 (끝에 / 없이)

if "내프로젝트" in DEPLOY_URL:
    print("★ DEPLOY_URL 에 내 Vercel 주소를 먼저 넣으세요")
else:
    call("GET", "/", base=DEPLOY_URL)
    call("POST", "/explain", {"word": "배포"}, base=DEPLOY_URL)

GET / → 상태 번호 200
{
  "message": "AI 용어 설명 API 입니다. /docs 에서 사용법을 보세요."
}
POST /explain → 상태 번호 401
{
  "detail": "접근 코드가 틀렸습니다 — 헤더 X-Access-Code 를 확인하세요"
}


> 👀 **직접 확인** — 상태 번호 503 이 나오면? (환경 변수를 넣고 **Redeploy** 했는지) · 휴대폰 브라우저로 `…vercel.app/docs` 를 열어 보세요.

In [22]:
# ============================================================
# [실습 5-6] 키 노출 점검 — 내 GitHub 저장소 파일에 키가 없는지 확인 (요청 없음)
# ============================================================
GITHUB_ID = "psy0635-ctrl"              # ★ (직접 작성) GitHub 아이디
REPO_NAME = "my-ai-api"             # ★ (직접 작성) 저장소 이름

my_key = userdata.get("GEMINI_API_KEY")
for file_name in ["main.py", "requirements.txt", ".env"]:
    url = f"https://raw.githubusercontent.com/{GITHUB_ID}/{REPO_NAME}/main/{file_name}"
    response = requests.get(url, timeout=30)
    if response.status_code == 404:
        print(f"·  {file_name} — 저장소에 없음" + (" (정상 — 올리면 안 되는 파일)" if file_name == ".env" else " → 아이디 · 저장소 이름 확인"))
    elif my_key in response.text or "AIza" in response.text:
        print(f"❌ {file_name} 에 키가 들어 있음 — 파일을 고치고, AI Studio 에서 키를 삭제 후 새로 발급하세요")
    else:
        print(f"✅ {file_name} — 키 없음")

✅ main.py — 키 없음
✅ requirements.txt — 키 없음
·  .env — 저장소에 없음 (정상 — 올리면 안 되는 파일)


---
# STEP 6. 심화 ① 접근 코드로 내 API 지키기 (실습 G · 약 20분)

배포한 API 는 **주소를 아는 누구나** 부를 수 있고, 그때마다 **내 키의 무료 한도**가 줄어듭니다.
**접근 코드**는 요청 헤더 `X-Access-Code` 에 정해 둔 값을 실어 보낸 요청만 통과시키는 간단한 열쇠입니다 (값은 서버 환경 변수 `APP_ACCESS_CODE` 에만 둠).

In [19]:
# 📋 main.py · [실습 6-1] 심화 ① 접근 코드로 내 API 지키기 — 헤더 X-Access-Code (POST 한 번에 요청 1번)
import os
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"
MAX_TURNS = 6                                 # 최근 몇 턴을 기억할지
SYSTEM_PROMPT = "너는 '폴리봇'이야. 인공지능소프트웨어과 1학년의 파이썬 · AI 공부 도우미야. 존댓말로 3문장 이내로 답한다. 개인정보를 묻지 않는다."
app = FastAPI(title="AI 용어 설명 API", description="용어를 보내면 Gemini 가 쉬운 말로 설명해 줍니다.")


class Question(BaseModel):
    word: str = Field(min_length=1, max_length=30, description="궁금한 용어")
    style: str = Field(default="초등학생도 알아듣게 쉽게", description="설명 말투")


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=500, description="이번에 보낼 말")
    history: list[dict] = Field(default_factory=list, description="지난 대화 기록 (이전 응답의 history 를 그대로 보냄)")


class ChatReply(BaseModel):
    answer: str
    history: list[dict]                       # 다음 요청 때 그대로 다시 보낼 기록


class Answer(BaseModel):                      # 응답(JSON)의 모양 — /docs 에도 그대로 나옴
    word: str
    answer: str
    tokens: int | None = None


def get_client():
    """키는 서버의 환경 변수 GEMINI_API_KEY 에서 읽음 — 코드 · 저장소에 적지 않음"""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise HTTPException(status_code=503, detail="서버에 GEMINI_API_KEY 가 없습니다 — 환경 변수를 확인하세요")
    return genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=60_000, retry_options=types.HttpRetryOptions(http_status_codes=[500, 503])),
    )


def check_access(access_code):
    """서버 환경 변수 APP_ACCESS_CODE 가 있으면, 요청 헤더의 접근 코드가 같아야만 통과"""
    expected = os.environ.get("APP_ACCESS_CODE")
    if expected and access_code != expected:
        raise HTTPException(status_code=401, detail="접근 코드가 틀렸습니다 — 헤더 X-Access-Code 를 확인하세요")


def trim_history(history, max_turns):
    """최근 max_turns 턴만 남긴 새 리스트를 돌려준다 (11주차와 같음)"""
    user_positions = [index for index, step in enumerate(history) if step.get("type") == "user_input"]
    if len(user_positions) <= max_turns:
        return history
    return history[user_positions[-max_turns]:]


def ask_gemini(client, system_instruction, contents):
    """Gemini 에 묻고, 실패하면 알맞은 상태 번호와 한글 안내로 바꿔 돌려보냄"""
    try:
        return client.interactions.create(model=MODEL_NAME, store=False, system_instruction=system_instruction, input=contents)
    except Exception as error:
        code = getattr(error, "status_code", None)
        if code == 429:
            raise HTTPException(status_code=429, detail="Gemini 무료 한도를 넘었습니다 — 1분 뒤 다시 요청하세요")
        raise HTTPException(status_code=502, detail=f"Gemini 요청 실패 (상태 번호 {code})")


@app.get("/")
def home():
    return {"message": "AI 용어 설명 API 입니다. /docs 에서 사용법을 보세요."}


@app.post("/explain", response_model=Answer)
def explain(question: Question, x_access_code: str | None = Header(default=None)):   # 헤더 X-Access-Code 를 받음
    check_access(x_access_code)               # ★ Gemini 를 부르기 전에 먼저 검사 (틀리면 요청이 나가지 않음)
    client = get_client()
    reply = ask_gemini(client, f"너는 AI 용어 설명 도우미야. {question.style} 3문장 이내로 설명한다.", question.word)
    return Answer(word=question.word, answer=reply.output_text, tokens=reply.usage.total_tokens)


@app.post("/chat", response_model=ChatReply)
def chat(request: ChatRequest, x_access_code: str | None = Header(default=None)):
    """서버는 대화를 기억하지 않음 — 부르는 쪽이 history 를 들고 와서 다시 보냄 (11주차 stateless 방식)"""
    check_access(x_access_code)
    client = get_client()
    send_list = trim_history(request.history + [{"type": "user_input", "content": [{"type": "text", "text": request.message}]}], MAX_TURNS)
    reply = ask_gemini(client, SYSTEM_PROMPT, send_list)
    return ChatReply(answer=reply.output_text, history=send_list + [step.model_dump(mode="json") for step in reply.steps])

✅ main.py 저장 · API 서버 켜짐 — 만든 주소 :
   · GET   /
   · POST  /explain
   · POST  /chat
ℹ️ 서버는 Colab 컴퓨터 안(localhost:8000)에 있음 — 내 PC 브라우저 주소창으로는 열리지 않음 → 아래 🧪 셀의 call() 로 부르기
👉 배포할 때는 이 셀 전체를 복사(Ctrl+A → Ctrl+C) → GitHub 의 main.py 에 붙여 넣기 → Commit


In [24]:
# 🧪 [실습 6-2] 접근 코드를 정하고 서버를 다시 켠 뒤 시험 (요청 1번)
ACCESS_CODE = "6IVvtPHFuDgfFehbWfb_rQE0OIoL2oRKVOohEa5kF5o"        # ★ (직접 작성) 내가 정하는 접근 코드 — 영어 · 숫자 · 하이픈만 · 남에게 알려 주지 않음
start_server()                      # 환경 변수 APP_ACCESS_CODE 를 넣어 서버를 다시 켬

call("POST", "/explain", {"word": "헤더"})                                              # 코드 없이 → 401
call("POST", "/explain", {"word": "헤더"}, headers={"X-Access-Code": "wrong-code"})      # 틀린 코드 → 401
# ※ 헤더 값에는 영어 · 숫자 · 하이픈만 씀 (한글을 넣으면 requests 가 UnicodeEncodeError)
call("POST", "/explain", {"word": "헤더"}, headers={"X-Access-Code": ACCESS_CODE})      # 맞는 코드 → 200

POST /explain → 상태 번호 401
{
  "detail": "접근 코드가 틀렸습니다 — 헤더 X-Access-Code 를 확인하세요"
}
POST /explain → 상태 번호 401
{
  "detail": "접근 코드가 틀렸습니다 — 헤더 X-Access-Code 를 확인하세요"
}
POST /explain → 상태 번호 200
{
  "word": "헤더",
  "answer": "'헤더'는 편지의 맨 위에 쓰는 편지머리처럼, **컴퓨터 문서나 웹사이트의 가장 맨 위에 들어가는 제목이나 인사말** 같은 공간이야. \n주로 글의 제목이나 만든 사람, 혹은 중요한 정보가 먼저 보이도록 윗부분에 쏙 넣어둔단다.\n예를 들어, 공책의 맨 윗줄에 날짜와 이름을 적어두는 것과 비슷한 역할이야!",
  "tokens": 128
}


<Response [200]>

> 👀 **직접 확인** — 401 일 때 Gemini 요청이 나갔을까요? (힌트: `check_access` 가 `get_client` 보다 먼저)

### 실습 6-3 — 배포에도 적용하기
1. 위 📋 셀 전체를 GitHub `main.py` 에 붙여 넣고 **Commit**
2. Vercel → **Settings → Environment Variables** → **Add Environment Variable** → Key `APP_ACCESS_CODE` · Value 내가 정한 코드 → **Save** → **Redeploy** (처음에는 목록에 없음 — 새로 추가)
3. `/docs` 에서 **Try it out** 하면 `x-access-code` 칸이 생깁니다 — 비워 두면 401, 코드를 넣으면 200

---
# STEP 7. 화면(Streamlit)과 배포한 API(Vercel) 연결 (실습 H · 약 20분)

**프런트엔드**는 사람이 보는 화면, **백엔드**는 화면 뒤에서 일하는 서버입니다.
오늘 만든 API(Vercel)를 백엔드로, 12주차 Streamlit 앱(Community Cloud)을 프런트엔드로 이어 붙입니다.

`https://….streamlit.app` (화면) → `https://….vercel.app/explain` (내 API) → Gemini

| 서비스 | 설정 위치 | 넣는 값 | 쓰는 파일 |
|---|---|---|---|
| Vercel (백엔드) | 프로젝트 → Settings → Environment Variables | `GEMINI_API_KEY` · `APP_ACCESS_CODE` | `main.py` |
| Streamlit Community Cloud (프런트엔드) | share.streamlit.io → 앱 ⋮ → Settings → Secrets | `API_URL` · `APP_ACCESS_CODE` | `app.py` |

- Gemini 키는 **Vercel 에만** 둡니다. 화면(app.py)은 Gemini 를 부르지 않으므로 키가 필요 없습니다.
- `APP_ACCESS_CODE` 는 **두 곳에 같은 값**을 넣습니다 (화면이 헤더 `X-Access-Code` 로 보내고, 서버가 비교).

In [25]:
# 📋 app.py · [실습 7-1] 화면(Streamlit · 프런트엔드)이 배포한 API(Vercel · 백엔드)를 부름 — 접근 코드를 헤더에 실어 보냄 (설명 한 번에 요청 1번)
# · Community Cloud 의 Secrets 에 API_URL · APP_ACCESS_CODE 두 줄을 넣음 (Gemini 키는 넣지 않음 — 키는 Vercel 에만)
import requests
import streamlit as st

try:
    API_URL = st.secrets["API_URL"]                  # Secrets 의 이름(API_URL)으로 꺼냄 — 대괄호 안에 주소를 쓰지 않음
except Exception:
    API_URL = "http://localhost:8000"                # Secrets 가 없으면(Colab 점검) 같은 런타임의 API 서버

try:
    ACCESS_CODE = st.secrets["APP_ACCESS_CODE"]      # 접근 코드도 Secrets 에서만 읽음 (코드 · GitHub 에 적지 않음)
except Exception:
    ACCESS_CODE = ""                                 # 접근 코드를 쓰지 않는 서버면 빈 값

st.title("📝 AI 용어 설명기 (API 버전)")
st.caption(f"API 주소 : {API_URL}")

style = st.selectbox("말투", ["초등학생도 알아듣게 쉽게", "전공 수업처럼 정확하게", "재미있는 비유를 들어"])
word = st.text_input("궁금한 용어", placeholder="예: API · 토큰 · 서버")

if st.button("설명 듣기", disabled=(word.strip() == "")):
    with st.spinner("API 에 묻는 중..."):
        try:
            response = requests.post(
                f"{API_URL}/explain",
                json={"word": word, "style": style},
                headers={"X-Access-Code": ACCESS_CODE},  # 서버의 APP_ACCESS_CODE 와 같아야 200 · 다르면 401
                timeout=90,
            )
        except requests.RequestException:
            st.error("❌ API 서버에 닿지 못함 — 화면의 'API 주소' 가 vercel.app 주소인지 확인하세요.")
        else:
            if response.status_code == 200:
                data = response.json()
                st.markdown(data["answer"])
                st.caption(f"상태 번호 200 · 토큰 {data['tokens']}")
            else:
                st.error(f"❌ 상태 번호 {response.status_code} : {response.json().get('detail')}")

✅ 오류 없음 — app.py 로 저장했습니다. 처음 화면 : ['📝 AI 용어 설명기 (API 버전)'] / 입력칸 : ['궁금한 용어'] / 버튼 : ['설명 듣기']


In [26]:
# ============================================================
# 🧪 [실습 7-2] Community Cloud 의 Secrets 를 흉내 내어 점검 — 화면 → 내 API → Gemini (요청 1번)
# ------------------------------------------------------------
# app_test.secrets[...] 가 Community Cloud 의 Secrets 칸 역할을 합니다
# ============================================================
API_FOR_TEST = "http://localhost:8000"    # ★ 배포 뒤에는 "https://내프로젝트.vercel.app" 로 바꿔 다시 실행해도 됨

for code_value in (["", ACCESS_CODE] if ACCESS_CODE else [""]):     # 접근 코드 없이 → 401 · 맞는 코드 → 200
    app_test = AppTest.from_file(os.path.abspath("app.py"), default_timeout=120)
    app_test.secrets["API_URL"] = API_FOR_TEST
    app_test.secrets["APP_ACCESS_CODE"] = code_value
    app_test.run()
    app_test.text_input[0].input("서버").run()
    app_test.button[0].click().run()
    result = app_test.error[0].value if app_test.error else app_test.markdown[-1].value[:80]
    print("접근 코드", "없음" if code_value == "" else "있음", "|", app_test.caption[0].value, "|", result)

접근 코드 없음 | API 주소 : http://localhost:8000 | 상태 번호 401 : 접근 코드가 틀렸습니다 — 헤더 X-Access-Code 를 확인하세요
접근 코드 있음 | API 주소 : http://localhost:8000 | 서버는 친구들과 게임을 하거나 유튜브를 볼 때, 우리가 필요한 정보를 척척 보내주는 아주 똑똑한 **큰 컴퓨터**야. 내 스마트폰이나 컴퓨터(손


> 👀 **직접 확인** — 접근 코드 없이 보낸 첫 번째는 왜 401 일까요? 그때 Gemini 요청이 나갔을까요?

### 실습 7-3 — Community Cloud 에 올리고 Secrets 두 줄 넣기
1. 12주차 실습 저장소(`streamlit-lab`)의 `app.py` → ✏️ → 위 📋 셀 **코드 안을 클릭** → Ctrl+A → Ctrl+C → 붙여 넣기 → **Commit changes**
2. <https://share.streamlit.io> → 내 앱 줄 오른쪽 **⋮ → Settings → Secrets** → 아래 두 줄 → **Save**
   (앱 화면 오른쪽 아래 **Manage app** 에서 들어가도 같은 설정 화면)
   ```toml
   API_URL = "https://내프로젝트.vercel.app"
   APP_ACCESS_CODE = "Vercel 에 넣은 것과 같은 코드"
   ```
   - `API_URL` 은 Vercel 프로젝트의 **Domains** 에 있는 짧은 주소 · 끝에 `/` · `/docs` · `/explain` 을 붙이지 않음
   - 접근 코드는 영어 · 숫자 · 하이픈만 (한글은 헤더 오류)
3. 앱 새로 고침 → 화면의 **API 주소** 가 `https://….vercel.app` 인지 확인 → 용어 입력 → **설명 듣기**

### 자주 하는 실수
| 화면에 보이는 것 | 원인 | 고치는 법 |
|---|---|---|
| API 주소 : `http://localhost:8000` + ❌ API 서버에 닿지 못함 | Secrets 에 `API_URL` 이 없음 · 이름 오타 · `st.secrets["https://…"]` 처럼 대괄호 안에 주소를 씀 | 대괄호 안은 이름 `"API_URL"` · 주소는 Secrets 칸에 |
| ❌ 상태 번호 401 : 접근 코드가 틀렸습니다 | Secrets 의 `APP_ACCESS_CODE` 가 없거나 Vercel 값과 다름 · app.py 에 `headers=` 가 없음 | 두 곳의 값을 똑같이 · 실습 7-1 코드로 교체 |
| ❌ 상태 번호 503 : 서버에 GEMINI_API_KEY 가 없습니다 | Vercel 환경 변수에 키 없음 · Redeploy 안 함 | 실습 5-3 |
| ❌ 상태 번호 404 | `API_URL` 끝에 `/explain` · `/docs` 를 붙임 | 짧은 주소만 |

---
# 📝 오늘의 과제 — 나만의 AI API(백엔드)를 배포하고 Streamlit 화면(프런트엔드)과 연결하기

### 조건
1. `SYSTEM_PROMPT` 에 내 주제의 여섯 칸(이름 · 역할 / 말투 / 할 일 / 하지 않을 일 / 모를 때 / 개인정보)을 씀
2. 주소 3개 — `GET /` (소개) · `GET /examples` (예시 질문) · `POST /ask` (질문 → 답, `tone` 으로 말투 선택)
3. `POST /ask` 는 **접근 코드** 없으면 401
4. 새 저장소(`my-api-주제`)에 `main.py` · `requirements.txt` 만 — **키 · 접근 코드는 없음** (실습 5-6 점검 ✅)
5. Vercel 에 배포 · 환경 변수 2개(`GEMINI_API_KEY` · `APP_ACCESS_CODE`) · `/docs` 가 열림
6. 아래 🧪 셀로 **배포 주소**를 불러 결과가 출력됨
7. **Streamlit 화면에서 내 API 를 불러 씀** — Community Cloud Secrets 에 `API_URL` · `APP_ACCESS_CODE` 두 줄 · 화면에서 답이 나옴 (상태 번호 200)

### 과제 1단계 — Colab 에서 완성
아래 📋 셀의 ★ (직접 작성) 부분을 고치고 실행 → 🧪 셀로 시험

In [ ]:
# 📋 main.py · 과제 — 나만의 AI API
# ============================================================
# 13주차 오늘의 과제 · 나만의 AI API (main.py)
# ------------------------------------------------------------
# · ★ (직접 작성) 표시가 붙은 곳만 내 주제로 바꾸면 됨
# · 키는 서버 환경 변수 GEMINI_API_KEY 로만 읽음 (Vercel → Settings → Environment Variables)
# · 접근 코드는 환경 변수 APP_ACCESS_CODE — 부르는 쪽은 헤더 X-Access-Code 에 같은 값을 넣음
#   ★ 이 파일 · GitHub 저장소에 키와 접근 코드를 절대 적지 않음
# · POST /ask 한 번에 Gemini 요청 1번 (무료 한도 분당 15회)
# ============================================================
import os
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"

# ★ (직접 작성) ① API 이름 · 한 줄 소개
APP_TITLE = "레시피 API"
APP_DESCRIPTION = "자취하는 대학생에게 간단한 요리를 알려 주는 AI API"

# ★ (직접 작성) ② 시스템 프롬프트 — 11주차 설계표 여섯 칸 · {tone} 자리에는 요청의 말투가 들어감 (지우지 말 것)
SYSTEM_PROMPT = """너는 '레시피봇'이야. 자취하는 대학생에게 간단한 요리를 알려 주는 도우미야.
규칙 1. {tone} 말투로 4문장 이내로 답한다.
규칙 2. 재료 · 조리 순서 · 걸리는 시간을 알려 준다.
규칙 3. 요리와 관련 없는 질문에는 '저는 요리 도우미라서 그 질문에는 답하기 어려워요.' 라고 답한다.
규칙 4. 가게 가격 · 영업시간처럼 모르는 정보는 지어내지 말고 직접 확인하라고 안내한다.
규칙 5. 이름 · 학번 · 전화번호 같은 개인정보를 묻지 않는다."""

# ★ (직접 작성) ③ 예시 질문 3개 (GET /examples 가 돌려줌)
EXAMPLE_QUESTIONS = ["계란으로 10분 안에 만들 요리?", "전자레인지만 있으면 뭘 만들 수 있어?", "남은 밥으로 만들 수 있는 요리?"]

app = FastAPI(title=APP_TITLE, description=APP_DESCRIPTION)


class AskRequest(BaseModel):
    question: str = Field(min_length=1, max_length=300, description="질문")
    tone: str = Field(default="친근한 존댓말", max_length=20, description="답의 말투")


class AskReply(BaseModel):
    question: str
    answer: str
    tokens: int | None = None


def check_access(access_code):
    """APP_ACCESS_CODE 가 정해져 있으면, 헤더의 접근 코드가 같아야만 통과"""
    expected = os.environ.get("APP_ACCESS_CODE")
    if expected and access_code != expected:
        raise HTTPException(status_code=401, detail="접근 코드가 틀렸습니다 — 헤더 X-Access-Code 를 확인하세요")


def get_client():
    """키는 서버의 환경 변수 GEMINI_API_KEY 에서 읽음"""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise HTTPException(status_code=503, detail="서버에 GEMINI_API_KEY 가 없습니다 — 환경 변수를 확인하세요")
    return genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=60_000, retry_options=types.HttpRetryOptions(http_status_codes=[500, 503])),
    )


@app.get("/")
def home():
    return {"name": APP_TITLE, "description": APP_DESCRIPTION, "how_to_use": "POST /ask 에 {\"question\": \"…\"} 를 보내세요. 자세한 사용법은 /docs"}


@app.get("/examples")
def examples():
    return {"examples": EXAMPLE_QUESTIONS}


@app.post("/ask", response_model=AskReply)
def ask(request: AskRequest, x_access_code: str | None = Header(default=None)):
    check_access(x_access_code)
    client = get_client()
    try:
        reply = client.interactions.create(
            model=MODEL_NAME, store=False,
            system_instruction=SYSTEM_PROMPT.format(tone=request.tone),
            input=request.question,
        )
    except Exception as error:
        code = getattr(error, "status_code", None)
        if code == 429:
            raise HTTPException(status_code=429, detail="Gemini 무료 한도를 넘었습니다 — 1분 뒤 다시 요청하세요")
        raise HTTPException(status_code=502, detail=f"Gemini 요청 실패 (상태 번호 {code})")
    return AskReply(question=request.question, answer=reply.output_text, tokens=reply.usage.total_tokens)

In [ ]:
# 🧪 [과제] 내 API 시험 — 로컬(Colab) 또는 배포 주소 (요청 1번)
BASE = "http://localhost:8000"      # ★ 배포 후에는 "https://내프로젝트.vercel.app" 로 바꿔 다시 실행
call("GET", "/", base=BASE)
call("GET", "/examples", base=BASE)
call("POST", "/ask", {"question": "계란으로 10분 안에 만들 요리 알려 줘.", "tone": "씩씩한 반말"},
     headers={"X-Access-Code": ACCESS_CODE}, base=BASE)
call("POST", "/ask", {"question": "접근 코드 없이 보내면?"}, base=BASE)     # → 401 (요청 없음)

### 과제 2단계 — 배포 · 점검
1. 새 GitHub 저장소 → `requirements.txt` · `main.py`(위 📋 셀) → Vercel **Import → Deploy**
2. 환경 변수 `GEMINI_API_KEY` · `APP_ACCESS_CODE` → **Redeploy**
3. 위 🧪 셀의 `BASE` 를 배포 주소로 바꿔 다시 실행 · 실습 5-6 점검(`REPO_NAME` 변경)

### 과제 3단계 — 화면(Streamlit)과 연결
아래 📋 app.py 셀은 **고칠 곳이 없습니다.** 화면 이름 · 소개 · 예시 질문은 내 API 의 `GET /` · `GET /examples` 에서 가져옵니다.

In [ ]:
# 📋 app.py · [과제 3단계] 나만의 AI API 를 부르는 화면 (프런트엔드) — 고칠 곳 없음 · Secrets 의 API_URL · APP_ACCESS_CODE 만 넣으면 됨 (질문 한 번에 요청 1번)
import requests
import streamlit as st

try:
    API_URL = st.secrets["API_URL"]                  # Community Cloud Secrets — 내 API 의 짧은 vercel.app 주소
except Exception:
    API_URL = "http://localhost:8000"                # Secrets 가 없으면(Colab 점검) 같은 런타임의 API 서버

try:
    ACCESS_CODE = st.secrets["APP_ACCESS_CODE"]      # Vercel 의 APP_ACCESS_CODE 와 같은 값 (코드 · GitHub 에 적지 않음)
except Exception:
    ACCESS_CODE = ""

try:                                                 # GET / · GET /examples — 이름 · 소개 · 예시 질문 (Gemini 요청 없음)
    info = requests.get(f"{API_URL}/", timeout=30).json()
    examples = requests.get(f"{API_URL}/examples", timeout=30).json()["examples"]
except (requests.RequestException, ValueError, KeyError):
    st.error("❌ API 서버에 닿지 못함 — 아래 'API 주소' 가 내 vercel.app 주소인지 확인하세요.")
    st.caption(f"API 주소 : {API_URL}")
    st.stop()

st.title(info.get("name", "나만의 AI"))
st.caption(f"{info.get('description', '')} · API 주소 : {API_URL}")

tone = st.selectbox("말투", ["친근한 존댓말", "씩씩한 반말", "정중한 안내문"])
question = st.text_input("질문", placeholder=examples[0] if examples else "")
if examples:
    st.caption("예시 질문 : " + " / ".join(examples))

if st.button("물어보기", disabled=(question.strip() == "")):
    with st.spinner("API 에 묻는 중..."):
        try:
            response = requests.post(
                f"{API_URL}/ask",
                json={"question": question, "tone": tone},
                headers={"X-Access-Code": ACCESS_CODE},
                timeout=90,
            )
        except requests.RequestException:
            st.error("❌ API 서버에 닿지 못함 — 잠시 뒤 다시 눌러 보세요.")
        else:
            if response.status_code == 200:
                data = response.json()
                st.markdown(data["answer"])
                st.caption(f"상태 번호 200 · 토큰 {data['tokens']}")
            else:
                st.error(f"❌ 상태 번호 {response.status_code} : {response.json().get('detail')}")

In [ ]:
# 🧪 [과제 3단계] 화면에서 내 API 부르기 — Secrets 흉내 (요청 1번)
app_test = AppTest.from_file(os.path.abspath("app.py"), default_timeout=120)
app_test.secrets["API_URL"] = BASE                       # 배포 뒤에는 BASE 가 내 vercel.app 주소
app_test.secrets["APP_ACCESS_CODE"] = ACCESS_CODE or ""
app_test.run()
print("화면 제목 :", app_test.title[0].value if app_test.title else app_test.error[0].value)
app_test.text_input[0].input("계란으로 10분 안에 만들 요리 알려 줘.").run()
app_test.button[0].click().run()
print(app_test.error[0].value if app_test.error else app_test.markdown[-1].value[:120])
print(app_test.caption[-1].value)

1. 새 GitHub 저장소(`my-app-주제`) → `app.py`(위 📋 셀) · `requirements.txt`(`streamlit==1.66.1` 한 줄) → Commit
2. <https://share.streamlit.io> → **Create app** → 저장소 · `app.py` 선택 → **Advanced settings → Secrets** 에 두 줄 → **Deploy**
   ```toml
   API_URL = "https://내프로젝트.vercel.app"
   APP_ACCESS_CODE = "Vercel 에 넣은 것과 같은 코드"
   ```
3. 휴대폰으로 `https://….streamlit.app` 을 열어 질문 → 답과 '상태 번호 200' 확인 (안 되면 실습 7 의 '자주 하는 실수' 표)

### ✏️ 정리 칸 (직접 작성)
- 내 API 주소 (`https://….vercel.app`) :
- 내 화면 주소 (`https://….streamlit.app`) :
- `/docs` 에서 본 주소(엔드포인트) 3개 :
- 접근 코드 없이 보냈을 때 상태 번호 :
- GitHub 저장소 주소 :
- 프런트엔드(Streamlit)와 백엔드(Vercel)에 각각 무엇을 넣었나 (키 · 주소 · 접근 코드) :

---
# 👥 3교시 · 팀 프로젝트 기획 메모 (제출 아님 · 팀장 구두 발표 준비용)

팀장이 **2분 안에** 아래 다섯 가지를 말합니다. 팀원과 함께 채워 두면 발표가 쉬워집니다.

| 발표할 것 | 메모 |
|---|---|
| ① 팀 이름 · 팀원 · 역할 (팀장 · 모델/API · 화면 · 발표) |  |
| ② 주제 — 누구의 어떤 불편을 해결하나 (한 문장) |  |
| ③ 쓸 기술 2가지 이상과 흐름 (입력 → 처리 → 출력) |  |
| ④ 배포 방법 · 누구 계정의 키로 돌리나 |  |
| ⑤ 15주차 발표 때 시연할 화면 (입력 → 답) |  |

---
## 📚 더 알아보기 (2026-11-09 접속 확인)
| 자료 | 주소 |
|---|---|
| FastAPI 공식 자습서 — 첫걸음 (한국어) | <https://fastapi.tiangolo.com/ko/tutorial/first-steps/> |
| Vercel — FastAPI 배포 안내 | <https://vercel.com/kb/guide/ship-a-fastapi-app-on-vercel> |
| Vercel — 환경 변수 관리 | <https://vercel.com/docs/environment-variables/managing-environment-variables> |
| Vercel — Hobby 플랜 (무료 조건) | <https://vercel.com/docs/plans/hobby-plan> |
| Streamlit Community Cloud — Secrets 관리 | <https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/secrets-management> |
| FastAPI Cloud (2026-11-02 정식 출시 · 참고) | <https://fastapicloud.com/> |